***
Test file for checking if functions are called properly
***
<div class="alert alert-block alert-info">
<b>v2 version of demo notebook will try "import" python files instead of writing everything in the cells</b>
</div>


In [10]:
#GlycoMSParser 0.3
#Import cell
GlycoMSP_version = 0.3
GlycoMSP_date = 20231007
GlycoMSP_build = "manv2"

In [ ]:
#GlycoMSParser 0.3
GlycoMSP_version_extractor = 0.2
GlycoMSPinit = True #To indicate cells below if the read process has been completed

#File conversion cell

try:
    from pymsfilereader import MSFileReader
except:
    print("Please run this code on a Windows-based OS with pymsfilereader and Thermo library installed")
#debug
def debug_readraw(debug=False):
    #rawfile.Close()
    print("[TEST]raw file has been closed successfully.")
#end of debug function

#init

try:
    print("GlycoMSP uses tkinter to get raw files") #change this to "to read GlycoMSP project dat, csv in supported formats and ms files" once dev is finished
    from tkinter import Tk
    from tkinter.filedialog import askopenfilename
    Tk().withdraw()
    filename = askopenfilename() #restrict to raw file, with text indications
    rawfile = MSFileReader(filename)
    print('File choosed', rawfile.Version())
    rawname = rawfile.GetFileName()
    print('The raw file you selected is: ', rawname)
except ImportError:
    raise ImportError('Please install tkinter to enable windows-supported file selection')

if not rawfile:
    try:
        rawfile = MSFileReader("G:\zf_sPerMeOG_intestine.raw")
        print('No tkinter detected. Select default raw file')
        print('Debug: raw file version: ', rawfile.Version())
        rawname = rawfile.GetFileName()
        print('GetFileName: ', rawname)
    except:
        print("Raw file read failed. Please check if the file path is correct.")

print("After importing raw file run the debug to break file handling")
#comment the debug functions below to avoid exit of file handling
#-----------------------------------------------#
#debug_readraw(debug=True)

In [ ]:
from collections import namedtuple
import time
import glob
import pandas as pd
import os
import pathlib

#copied from comparepeaklist.py
def calcproton(isolatedmass, charge):
    if abs(charge) == 1 :
        return isolatedmass
    elif charge >= 2:
        conv = (isolatedmass * charge - (charge - 1) * 1.00784)
        return conv
    elif charge == 0:
        return 0
    elif charge < -1:
        negconv = (isolatedmass * charge + (charge - 1) * 1.00784)
        return negconv
    

def peak_extractor(rawfile, auto = True, debug = False):

    print(f'This raw file has {rawfile.GetNumSpectra()} spectra')
    #define number of spectra to parse (TODO: make time exclusion convertible and need to be added to record file.)
    if not auto:
        scan_number = int(input('enter spectrum no you want to summarize'))
        maxn = int(rawfile.GetNumSpectra())
    else:
        scan_number = int(rawfile.GetNumSpectra())
        maxn = scan_number

    #init variables for storing parsed data
    ms1list, ms2list, ms3list, errlist = {}, {}, {}, {}
    ms1fromms2 = []
    b, c = [], []  #b for MS2, c for MS3
    header = ('entry no','MS1scan no', 'MS1Isolation mass', 'MS1monoIsomass','chargeState','in [H+]',
          'intensity','Structure', 'MS2 Scan no', 'peaklist')
    b.append(header)
    ms3header =  ('entry no','MS3scan no', 'MS2Isolation mass', 'MS2monoIsomass', 'MS2 Scan no', 'peaklist')
    c.append(ms3header)
    MS2peaklist = namedtuple('MS2peaklist', ('dMass', 'dIntensity'))
    MS3peaklist = namedtuple('MS3peaklist', ('dMass', 'dIntensity'))
    ms2count, ms3count = 1, 1, 

    #development area and exception prevention
    if debug:
        print("[debug] need to create debug logs when debug is set to True.")
        print("[debug]Also need a readable log that records needed information and the operation can be reproduced when loading that record.")
        print("[debug add debug log and record log in debug block]")
        print("[debug]: start calculate time spent for converting data")
        perf_esti1 = time.time()
    
    if scan_number > maxn:
        print(r"you're attempting to run a number more than the spectra this file has.")
        scan_number = maxn
        print('set scan_number to', maxn)
    #
    
    #perform file conversion in defined spectra range
    for i in range(scan_number):
        j = i+1
        if i > maxn:
            break
        elif rawfile.GetMSOrderForScanNum(j) == 1:
            #ms1list[i] = j
            pass
            #shooud I keep this? the list is totally referred from MS2
        elif rawfile.GetMSOrderForScanNum(j) == 2:
            peaklist = MS2peaklist((rawfile.GetLabelData(j)[0][0]), (rawfile.GetLabelData(j)[0][1]))
            isolationmass = rawfile.GetPrecursorInfoFromScanNum(j)[1]
            chargestate = rawfile.GetPrecursorInfoFromScanNum(j)[2]
            inHmass = calcproton(isolationmass,chargestate)
            a = (ms2count,
                rawfile.GetPrecursorInfoFromScanNum(j)[3],#parentScanNo, MS1
                rawfile.GetPrecursorInfoFromScanNum(j)[0],#Isolation mass
                isolationmass,#monoIsomass
                chargestate,#chargeState
                inHmass, #calculate calcproton(isolatedmass, charge)
                'ext from peak list',
                'structure na',
                j, #MS2scan no
                peaklist
                )
            b.append(a)
            ms2count +=1
        elif rawfile.GetMSOrderForScanNum(j) == 3:
            ms3list[i] = j
            peaklist = MS3peaklist((rawfile.GetMassListFromScanNum(j)[0][0]), (rawfile.GetMassListFromScanNum(j)[0][1]))
            d = (ms3count,
                rawfile.GetPrecursorInfoFromScanNum(j)[3],#parentScanNo
                rawfile.GetPrecursorInfoFromScanNum(j)[0],#Isolation mass
                rawfile.GetPrecursorInfoFromScanNum(j)[1],#monoIsomass
                j, #MS2scan no
                peaklist #surely will have error
                )
            c.append(d)
            ms3count +=1
        else:
            errlist[i] = j        
        i+=1
    print("Extraction finished.")
    rawfile.Close()

    if debug:
        extractiontime = time.time() - perf_esti1
        print("[debug]Raw file has been closed.")
        print("[debug]Preparing information for writing...")
        print(f"[debug]Time spent for extraction: {int(extractiontime)} seconds.")

    timestamp = time.strftime("%Y%m%d-%H%M%S") #datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

    #export MS2 spectra file in csv format
    save_ms2filename= input("Please enter the file name or leave it blank to generate a filename with datetime")
    if len(save_ms2filename) == 0:
        fileext = rawname + ' MS2 summary from ' + str(scan_number) + " at " + timestamp + '.csv'
    else:
        fileext = rawname + save_ms2filename + " at " + timestamp + ".csv"
    print(f"Filename output:{fileext}")
    perf_esti2 = time.time()
    with open(fileext, 'wt') as g:
        print("writing to csv...")
        for q in range(len(b)):
            print('\t'.join(map(str, (b[q]))), file = g)
    if debug:
        ms2writetime = time.time() - perf_esti2
        print("[debug]Finished writing MS2 information")
        print(f"[debug]Time spent for extraction: {int(ms2writetime)} seconds.")

    #export MS3 spectra file in csv format
    timestamp = time.strftime("%Y%m%d-%H%M%S") 
    save_ms3filename= input("Please enter the MS3 file name or leave it blank to generate a filename with datetime")
    if len(save_ms3filename) ==0 :
        fileext = rawname + ' MS3 summary from ' + str(scan_number) + " at " + timestamp + '.csv'
    else:
        fileext = rawname + save_ms3filename + " at " + timestamp + ".csv"
    print(f"Filename output:{fileext}")
    perf_esti3 = time.time()
    with open(fileext, 'wt') as h:
        print("writing to csv...")
        for q in range(len(c)):
            print('\t'.join(map(str, (c[q]))), file = h)
    if debug:
        ms3writetime = time.time() - perf_esti3
        print("[debug]Finished writing MS3 information")
        print(f"[debug]Time spent for extraction: {int(ms3writetime)} seconds.")


    #export log files...
    if debug:
        print("Write logs")
        print(f"file name: {fileext}")
        print(f"datetime (latest): {timestamp}")
        print(f"file name: {rawname}")
        print("file path is not possible in current context.")
        print(f"scan number set in this experiment: {scan_number}")
        print(f"GlycoMSP main version: {GlycoMSP_version}")
        #info in logs: datetime, filename, file path, scan number, version of GlycoMSP and EACH components
    
    #report errors
    print('err list')
    for key,value in errlist.items():
        print(value)

    print("dump finished")




def peak_exthandler(rawfile, filetype=None, debug=False):
    if filetype == "raw":
        peak_extractor(rawfile, debug)
    elif filetype == "mzML":
        print("mzML file will be supported in v1.1")
    else:
        print("unsupported format")


def runextractor(debug=False):
    if GlycoMSPinit:
        try:
            import glob
            import pandas as pd
            import os
            import pathlib
            import numpy as np
            import time  #for calculating efficiency
            #from collections import namedtuple
            import warnings
            warnings.simplefilter(action='ignore', category=FutureWarning) ###to supress warning
        except:
            print("Missing essential package for following analysis") #try to list out the package missing
        if rawfile:
            print("Start raw file extration process...")
            filetype="raw"
            peak_exthandler(rawfile, filetype, debug)

runextractor(debug=True)

In [4]:
def dummy_extractor(rawfile, auto = True, debug = False):
    if auto:
        print("Auto on")
    if debug:
        print("Debug on in extractor (passed by handler)")
    print("extractor")

def dummy_exthandler(rawfile, filetype=None, debug=False):
    if debug:
        print(f"debug is {debug}")
        print("Debug on in handler (passed by run)")
    if filetype == "raw":
        dummy_extractor(rawfile,auto = True, debug=debug)  #the debug isn't passed in the demo file (20231009)
    elif filetype == "mzML":
        print("mzML file will be supported in v1.1")
    else:
        print("unsupported format")

def dummyrun(debug=False):
    if debug:
        print("Debug in run")
    filetype="raw"
    rawfile = "test functionality"
    dummy_exthandler(rawfile, filetype, debug)

dummyrun(debug=True)

Debug in run
debug is True
Debug on in handler (passed by run)
Auto on
Debug on in extractor (passed by handler)
extractor


In [1]:
import csv
#import math
from decimal import Decimal, getcontext, ROUND_HALF_UP
from collections import namedtuple
#import re
import pandas as pd
import test_comparepeaklistppm
#import numpy as np
ppm = 50
inputlist = [123]
comparelist = [456]
inputintensitylist= [7890]
result = test_comparepeaklistppm.comparepeaklistppm(inputlist, comparelist, ppm, inputintensitylist)

inputlist is [123]
comparelist is [456]
inputintensitylist is [456]


In [2]:
import csv
#import math
from decimal import Decimal, getcontext, ROUND_HALF_UP
from collections import namedtuple
#import re
import pandas as pd
import comparepeaklist


LISTforOvary = [246.1336, 260.1492, 344.1704, 374.1810 ,376.1966, 406.2072, 432.2071, 450.2334, 464.2490, 505.2756, 580.2964, 621.3229, 638.3382,
                651.3335, 793.3808, 825.4227, 842.4380, 866.4492, 896.4598]



def comparepeaklistppm(inl = [], coml= [], ppm = 500, inil= []):
    inputlist = inl
    comparelist = coml[:]
    inputintensitylist = inil
    #pre-assign input list and compare list
    print(f"inputlist in another file is {inputlist}")
    print(f"comparelist in another file is {comparelist}")
    if (inputlist == []):
        print(f"inputlist in condition {inputlist}")
        inputlist = [344, 344.16, 344.19, 344.25, 345, 376, 432, 464]
        print('use default inputlist', inputlist)
    else:
        print('inputlist is', inputlist)
        pass
    if (comparelist == []):
        print(f"comparelist in condition {comparelist}")
        comparelist = [344.17, 374.18, 376.20, 406.21, 432.22, 450.23, 464.25, 580.30, 793.38, 825.42]
        print('use default comparelist', comparelist)
    else:
        print('comparelist is', comparelist)
        pass
    if (inputintensitylist == []):
        print(f"inputintensitylist in condition {inputintensitylist}")
        inputintensitylist = [500, 1000, 34567, 43210, 666666, 1, 810, 114514, 100000, 100000000000]
        print('use default comparelist', comparelist)
    else:
        print('inputintensitylist is', comparelist)
        pass
    #ppm = ppmcalculation()
    #normalize from inputintensitylist
    #pseudocode:
    #find largest value in inputintensitylist = maxin
    #ninputintensitylist = (inputintensitylist/maxin)*100 for i in range(inputintensitylist)
    maxin = max(inputintensitylist[:])
    #print('max', maxin)
    ninputintensitylist = []
    for i in range(len(inputintensitylist)):
        ninputintensitylist.append((inputintensitylist[i]/maxin)*100)
    #for test the normalized bug (fixed)
    '''
    if str(maxin)== str(3798419.75):
        print('hey', ninputintensitylist)
    else:
        pass
    '''
    hitcount = 0
    hitlist =[]
    location = []
    hitintensitylist = []
    nhitintensitylist = []
    for i in inputlist:
        #print('input', i)
        for j in comparelist:
            #print('comparing with', j)
            #find out which is larger
            if i > j:
                k = (i-j)/i
                ioverj = True
            elif i < j:
                k = (j-i)/j
                ioverj = False
            elif i == j:
                #test why 376 isn't appearing
                '''
                if i == 376.20:
                    print(i, 'found')
                ioverj = False
                '''  
                k = 0
            else:
                print('error in comparing i j')

            if (k * 1000000 < ppm):
                #location for finding intensity info
                loc = inputlist.index(i)
                location.append(loc)
                htmp = inputintensitylist[loc]
                ntmp = ninputintensitylist[loc]
                hitintensitylist.append(htmp)
                nhitintensitylist.append(ntmp)
                hitcount +=1
                #append j to prevent instrument readout errors resulting
                #in lots of redundent columns in small mass difference
                #2022.12.13 found bug (minor)
                hitlist.append(i)
                #print('it\'s hit')
                #print("now going to delete", comparelist[comparelist.index(j)])
                del comparelist[comparelist.index(j)]
                break
            elif (ioverj is True) and (k * 1000000 > ppm):
                #print('input > comparepeak, go up')
                #delete those never been compared in the coming iteration
                #print("now going to delete", comparelist[comparelist.index(j)])
                del comparelist[comparelist.index(j)]
                #print('delete those comparing ions smaller than input', comparelist)
            elif (ioverj is False) and (k *1000000 > ppm) :
                #print('input < comparepeak')
                break #byebye, check next i
            else:
                print('exception')
    return hitcount, hitlist, location, hitintensitylist, nhitintensitylist #, maxin

def pdms2test(comparelist1= None, debug=False):
    aaaaaaaaaaa = (comparelist1 == LISTforOvary)
    print(f"if the List of ovary is the same as input {aaaaaaaaaaa}")
    #print(f"comparelist is {comparelist}")
    df = pd.read_csv('zf_sPerMeNG_ovary.raw MS2 summary from 44510 at 20231007-235302.csv', sep='\t') # zf_sPerMeNG_intestine MS2 summary from 48127 at 20230611-182546 20230611-194107 predictedcomp.csv#'debug1.csv the MS2 summary from44218.csv'
    df.drop(columns=df.columns[0], axis=1, inplace=True) 
    #print(df.dtypes)
    err = 0
    notfound = 0
    hits = 0
    subdf = []
    ppm = comparepeaklist.ppmcalculation()
    debuglog = []
    tmpdf = pd.DataFrame()
    listinfo = True
    for i in range(3): #range(len(df)):
        comparelist = comparelist1
        aaaaaaaaaaa = (comparelist1 == LISTforOvary)
        bbbbbbbbbbb = (comparelist == LISTforOvary)
        cccccccc = (3==5)
        print(f"CNMB, {cccccccc}")
        print(f"if the List of ovary is the same as input {aaaaaaaaaaa}, {comparelist1}")
        print(f"if the List of ovary is the same as input in the loop {bbbbbbbbbbb}, {comparelist}")
        #initialize the placeholder for input peak information (to be parsed)
        #print(f"run {i} the comparelist is {comparelist}")
        inputlist = []
        inputintensitylist = []
        #parse peaklist
        #load no. i row's peaklist of the dataframe
        tmp = df.loc[i, "peaklist"]
        #the peak intensity and peak mass are included as well.
        inputlist1 = tmp.split('(')[2].split(')')[0].split(',') #peakmass
        d = tmp.split('(')[3].split(')')[0].split(',') #peakintensity
        #print('inputlist =',type(inputlist), inputlist1, 'intensity=',type(d), d) #debug
        #print('going to modify the number') #debug
        run = str(i+1) + " run"
        #debuglog.append(run)
        #debuglog.append("Original input")
        #debuglog.append(tmp)
        #debuglog.append("---------")
        #debuglog.append("Split input")
        #debuglog.append(inputlist1)
        print(f"length of input {len(inputlist1)}")
        for j in range(len(inputlist1)):
            k = Decimal((inputlist1[j])).quantize(Decimal("0.001"),rounding=ROUND_HALF_UP) 
            k = float(k)
            l = Decimal((d[j])).quantize(Decimal("0.001"),rounding=ROUND_HALF_UP)
            l = float(l)
            inputlist.append(k)
            inputintensitylist.append(l)
        #print(f"decimalized peak list: {inputlist}")
        #print(f"decimalized peak intensity {inputintensitylist}")
        rangetest = "in run " + str(i) +" the decimalized list is: "
        #debuglog.append(rangetest)
        #debuglog.append(inputlist)
        print(f"comapre list = {comparelist}")
        print(f"input list = {inputlist}")
        print(f"input intensity list = {inputintensitylist}")
        print("before compare")
        if comparelist is None:
            comparelist = [344.17, 374.18, 376.20, 406.21, 432.22, 450.23, 464.25, 580.30, 793.38, 825.42] #default one
        if listinfo is True:
            print('compare list=', comparelist)
            listinfo = False #let it only prints once

        #NO bugs until this part. No arguments were lost
        #forcomparelist = comparelist #to avoid deleting whole list... but why?
        #result = test_comparepeaklistppm.comparepeaklistppm(inputlist, comparelist, ppm, inputintensitylist)
        comparepeaklistppm(inputlist, comparelist, ppm, inputintensitylist)
        #result = comparepeaklistppm(inputlist, comparelist, ppm, inputintensitylist) #comparepeaklist.comparepeaklistppm
        #debuglog.append("Compare list:")
        #debuglog.append(comparelist)
        
        #debuglog.append(result[0])
        #debuglog.append(result[1])
        #debuglog.append(result[2])
        #debuglog.append(result[3])
        #debuglog.append(result[4])


        '''
        if result[0] > 0:
            df.loc[i, "in [H+]"] = comparepeaklist.calcproton(df.loc[i, "MS1Isolation mass"],df.loc[i, "chargeState"])
            #ms2scan = df.loc[i, "MS2 Scan no"]
            #hitspectrumlist.append([ms2scan, result[0], result[1], result[3], result[4]])
            hits+=1
            #before append calculate the charge conversion
            #protonateass = comparepeaklist.calcproton(df.loc[i, "MS1Isolation mass"], df.loc[i, "chargeState"])
            #I think I should add them back at thise stage
            #And theoretical mass
            #And prediction of composition if prediction flag is set to True)
            addinfo = pd.Series([result[1], result[3], result[4]], index=['hitpeaklist', 'selectedpeakintensity','normalizedselectedin']) #, result[5], 'maxintensity'
            tmpdf = pd.concat([df.loc[i], addinfo], axis = 0)
            subdf.append(tmpdf)
            #debug1 = int(df.loc[i, "MS2 Scan no"])
            #if debug1 == int(14857):    #test why 376 and the relative intensity value is wrong
            #    print(type(addinfo), type(subdf))
            #    print(addinfo, subdf)
            #    break
        elif result[0] == 0:
            #print('no hits')
            notfound+=1
        else:
            print('unexpected error')
            err+=1
            #do nothing
        #need to catch the returned value and decide if this spectra is ok
    print('hits', hits, 'notfound', notfound, 'err', err) #'hits', hitspectrumlist)
    '''
    if debug:
        with open("testcomparelist.txt", "a") as debuglogs:
                debuglogs.write(str(debuglog))

print()
pdms2test(LISTforOvary, debug=True)

Selected 

if the List of ovary is the same as input True
CNMB, False
if the List of ovary is the same as input True, [246.1336, 260.1492, 344.1704, 374.181, 376.1966, 406.2072, 432.2071, 450.2334, 464.249, 505.2756, 580.2964, 621.3229, 638.3382, 651.3335, 793.3808, 825.4227, 842.438, 866.4492, 896.4598]
if the List of ovary is the same as input in the loop True, [246.1336, 260.1492, 344.1704, 374.181, 376.1966, 406.2072, 432.2071, 450.2334, 464.249, 505.2756, 580.2964, 621.3229, 638.3382, 651.3335, 793.3808, 825.4227, 842.438, 866.4492, 896.4598]
length of input 38
comapre list = [246.1336, 260.1492, 344.1704, 374.181, 376.1966, 406.2072, 432.2071, 450.2334, 464.249, 505.2756, 580.2964, 621.3229, 638.3382, 651.3335, 793.3808, 825.4227, 842.438, 866.4492, 896.4598]
input list = [92.441, 92.649, 93.308, 96.373, 96.574, 97.039, 99.326, 101.976, 107.967, 109.404, 111.543, 112.961, 119.238, 124.65, 131.376, 133.255, 141.203, 158.132, 186.371, 192.944, 239.094, 258.076, 263.527, 277.922, 29

In [9]:
#pseudocode move to separated python file after function test
import csv
import pandas as pd



def def_spectralistheader(header, Version = None):
    #Defines if certain header name in specified version is needed or not.
    if Version is None:
        ver = "na"
    ver_orderedlist = ["na", "0.1"]  #find certain version to know it's order... should be automaticallt managed by some version manager tools, not manually
    print("Essential components: MS1scan no, MS1Isolation mass, MS1monoIsomass, chargeState, in [H+], intensity, Structure, MS2 Scan no, peaklist]")
    print("Files after processing will get 'relative intensity...' ")
    print("Files after prediction will get 'prediction dict......'")
    Essen_set = set('MS1scan no', 'in [H+]', 'MS2 Scan no', 'peaklist')
    Standard_set = set('entry no', 'MS1scan no', 'MS1Isolation mass', 'MS1monoIsomass', 'chargeState', 'in [H+]', 'intensity', 'Structure', 'MS2 Scan no', 'peaklist')
    # header intersect Essen_set  == Essen_set   -> meaning that header satisfies the need of it
    After_mass_prediction_set = set("???")
    After_structural_prediction_set = set("!!!")
    old_list = ["entry no", "MS1scan no",	"MS1Isolation mass",	"MS1monoIsomass",	"chargeState", 	"in [H+]", 	"intensityN\/A", 	"StructureN\/A", "MS2 Scan no",	"peaklist",	"Predicted composition"]


def pdreadcsv(dataframe):
    headers = dataframe.columns.tolist()
    print(headers)
    return headers


def datasetvalidation(loaddataset, stagedata=None):
    if stagedata:
        print("directly load dataset version if a log file is provided.")
    else:
        print("No stage data available. Try to guess the version from input file.")

    #load from csv or directly input a pandas dataframe
    if isinstance(loaddataset, pd.DataFrame):
        print("The dataset loading is a dataframe.")
        print("Validate column information...")
        pdreadcsv(loaddataset)
        #check header
    try:
        print("Trying to read the file as csv")
        with open(loaddataset, "r") as f:
            try:
                has_headings = csv.Sniffer().has_header(f.read(1024))
            except csv.Error:
                # The file seems to be empty
                has_headings = False
        print(f"Headers: {has_headings}")
    except:
        print("Not support formats")
    if has_headings:
        print("Get headers and validate them")
        #print(f"headers: {head}")
        dataframe = pd.read_csv(loaddataset, sep="\t")
        head = pdreadcsv(dataframe)
        return head



datasetvalidation("zf_sPerMeNG_ovary.raw MS2 summary from 44510 at 20231007-235302.csv")

No stage data available. Try to guess the version from input file.
Trying to read the file as csv
Headers: True
Get headers and validate them
['entry no', 'MS1scan no', 'MS1Isolation mass', 'MS1monoIsomass', 'chargeState', 'in [H+]', 'intensity', 'Structure', 'MS2 Scan no', 'peaklist']


['entry no',
 'MS1scan no',
 'MS1Isolation mass',
 'MS1monoIsomass',
 'chargeState',
 'in [H+]',
 'intensity',
 'Structure',
 'MS2 Scan no',
 'peaklist']

In [13]:
#stage data:

#headers
#get position or not? conpare in order or in set only?
def headervercheck(header):
    #print(header)
    #entry_h, ms1scan = -1
    #print(type(header))
    headerinfo = {}
    if "entry no" in header:
        entry_h = True
        entry_i = header.index("entry no")
        #get the location
        #for x, y in header, ref return index
        print(entry_i)
    if "MS1scan no" in header:
        ms1scan = True
        ms1scan_i = header.index("MS1scan no")
        print(ms1scan_i)
    #return a dict {colname: location} if not exist return -1 
    #consider ifi should store the info in dict NOT list
    headerinfo["version"] = 1.0


c = datasetvalidation("zf_sPerMeNG_ovary.raw MS2 summary from 44510 at 20231007-235302.csv")
#print(f"c is {type(c)}, {c}")
headervercheck(c)

No stage data available. Try to guess the version from input file.
Trying to read the file as csv
Headers: True
Get headers and validate them
['entry no', 'MS1scan no', 'MS1Isolation mass', 'MS1monoIsomass', 'chargeState', 'in [H+]', 'intensity', 'Structure', 'MS2 Scan no', 'peaklist']
0
1
